# Datamine EXTRA Process Example

This notebook demonstrates how to configure and run the **`extra`** process wrapper in `dmstudio` using practical, production-grade mining workflows.

## Process Description

**EXTRA** (EXpression TRAnslator) is a core Datamine process for data transformation. It evaluates expressions line-by-line across records to calculate new fields or modify existing ones using arithmetic, string manipulation, conditional statements, and sequential record context functions.

### Key Expression Syntax & Techniques (Learned from Production Scripts like `pymri_dm.py`)

When calling `dm_cmd.extra(in_i=..., out_o=..., expression=[...])`, pass your transformations as a Python list of strings. `dmstudio` formats each line and appends `'GO'` automatically.

1. **Explicit Field Creation & Data Types**:
   - **Numeric Field**: `FIELD_NAME;N = expression` (e.g. `LENGTH;N = TO - FROM`)
   - **Alphanumeric (String) Field**: `FIELD_NAME;A<length> = "string"` (e.g. `GRADE_CAT;A16 = "UNASSIGNED"`)
2. **Conditional Logic (`IF ... ELSEIF ... ELSE ... END`)**:
   - `IF (AU >= 3.0) GRADE_CAT = "HIGH_GRADE" ELSEIF (AU >= 0.5) GRADE_CAT = "MED_GRADE" ELSE GRADE_CAT = "LOW_GRADE" END`
3. **Sequential & Boundary Record Functions (`PREV`, `NEXT`, `FIRST`, `ABSENT`)**:
   - **Handle Missing Data**: `IF (AU == ABSENT()) AU = 0.0 END`
   - **Detect Drillhole/Group Boundaries**: `IF (FIRST()) IS_NEW_HOLE = 1 END` or `IF (BHID != PREV(BHID)) IS_NEW_HOLE = 1 END`
4. **Mathematical Calculations & Transformations**:
   - **Unit Conversions & Calculated Fields**: `AU_OPT;N = AU / 31.1035` or `LENGTH;N = TO - FROM`
5. **Intermediate Field Cleanup (`ERASE`)**:
   - **Delete Scratch/Temporary Fields**: `ERASE(TEMP_RATIO)` before writing the final output file.

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('extra')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments.

Here we define a multi-statement EXTRA expression demonstrating production techniques from practical mining workflows (`pymri_dm.py`):
- Explicit Numeric (` ;N `) and String (` ;A16 `) field creation
- Calculating missing interval fields (`LENGTH;N = TO - FROM`)
- Handling missing data with `ABSENT()`
- Multi-branch conditional logic (`IF ... ELSEIF ... ELSE ... END`)
- Boundary / sequence detection with `PREV()` and `FIRST()`
- Cleaning up scratch variables using `ERASE()`

In [ ]:
# Step 4: Execute extra process
# Expressions are passed as a list of strings to `dm_cmd.extra`.
# dmstudio automatically formats each line and appends 'GO'.

extra_expressions = [
    # 1. Explicit Field Declarations & Calculations (Numeric ;N and String ;A16)
    'LENGTH;N = TO - FROM',
    'AU_OPT;N = AU / 31.1035',
    'GRADE_CAT;A16 = "UNASSIGNED"',
    'FLAG_HIGH;N = 0',
    'IS_NEW_HOLE;N = 0',
    
    # 2. Handle missing / absent data
    'IF (AU == ABSENT()) AU = 0.0 END',
    
    # 3. Conditional Classification (Grade domaining)
    'IF (AU >= 3.0) GRADE_CAT = "HIGH_GRADE" ELSEIF (AU >= 0.5) GRADE_CAT = "MED_GRADE" ELSE GRADE_CAT = "LOW_GRADE" END',
    
    # 4. Binary flagging
    'IF (AU >= 3.0) FLAG_HIGH = 1 END',
    
    # 5. Drillhole boundary detection using FIRST() and PREV()
    'IF (FIRST()) IS_NEW_HOLE = 1 END',
    'IF (BHID != PREV(BHID)) IS_NEW_HOLE = 1 END',
    
    # 6. Intermediate scratch calculation & cleanup with ERASE()
    'TEMP_RATIO;N = AU / (LENGTH + 0.001)',
    'ERASE(TEMP_RATIO)'
]

print("Running extra process with multi-statement production expressions...")
dm_cmd.extra(
    in_i='t_assays',
    out_o='t_extra_out',
    expression=extra_expressions,
    approx_p=0,
    fldfail_p=1,
    print_p=0
)
print("extra execution completed.")

## Step 5: Verify Results
Check that output files exist on disk (`.dmx` or `.dm`) and read them using pandas to verify the newly created and transformed fields.

In [ ]:
# Step 5: Verify results
output_file = None
for candidate in ['t_extra_out.dmx', 't_extra_out.dm']:
    if os.path.exists(candidate):
        output_file = candidate
        break

if output_file:
    df = agent.read_datamine(output_file)
    print(f"Output file ({output_file}) loaded successfully. Rows: {len(df)}")
    
    # Display the key fields transformed by EXTRA
    display_cols = [col for col in ['BHID', 'FROM', 'TO', 'LENGTH', 'AU', 'AU_OPT', 'GRADE_CAT', 'FLAG_HIGH', 'IS_NEW_HOLE'] if col in df.columns]
    print("\nSample records showing transformed fields:")
    print(df[display_cols].head(10))
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")